這部分讀入套件，如果在 package 這個檔案修改了甚麼，再次執行這個cell就會更新。

In [1]:
import BigDataAnalysis_Package as bda
import importlib
importlib.reload(bda)

<module 'BigDataAnalysis_Package' from 'd:\\Code\\Python\\BigDataAnalysis_Package.py'>

總共有三個validation方法，分別是：
- validation:就是一般的validation，隨機拆分成指定比例的train和test，然後重複執行。
- kfold:把整個資料拆成 K 等分，一次拿一份做為測試集，其餘為訓練集，總共訓練並測試 K 次。
- stratified kfold:會讓拆分出來的資料比例與原資料相同。
假設原資料為

| 65 | 80 | 95 | 130 |
|----|----|----|-----|
| 10 | 20 | 30 | 40 |

以 K 是 10 為例，拆分出來的訓練集就會是

| 65 | 80 | 95 | 130 |
|----|----|----|-----|
| 9 | 18 | 27 | 36 |

與原資料中各類別間的比例相同， 1 : 2 : 3 : 4 。

三個驗證方法的參數基本相同，共同擁有的參數為
- data_dict : 資料集。
- K : 要重複驗證幾次，須注意 k fold 和 stratified k fold 的 K 同樣代表訓練集和測試集在資料中的比例。
- label : 資料是 'X' 還是 'Y'。
- pattern : 輸出結果如果是類別則要輸入 'discrete'，連續值為考慮空間座標 'continue'，及不考慮空間座標 'coord'。
- path : 訓練完的模型要存放的路徑。
- nperseg : 資料轉換成 PSD 及 CSD 時，所選擇的頻率的密集度，越高則代表選擇的頻率越多，數字須為2的冪次，預設為256。
- mask : 優些資料在高頻率的部分沒有差異，mask可以選擇要取用的頻率範圍，預設為 None，若要使用則輸入應為 (min_freq, max_freq)。
- clamp : 有時候利用模型進行訓練及預測時，若預測數值沒有給邊界限制，則預測的 y 可能會太發散，clamp 就是用來限制 y 的預測範圍，預設為 None，若要使用則輸入應為 (min_y, max_y)。
- val_label : 可以用來觀察每次進行驗證時，指定 label 的真實值和預測值，一次只能觀察一個 label，輸出是測試集的預測值，預設為 None，若要使用則輸入需要觀察的label。
- inner : 可以用來觀察每次進行 validation 時，所有 label 在這次驗證的 mse，預設為 False，改為 True 則會輸出所有 label 的 mse。

validation 這個指令有自己的參數 test_size，用來決定測試資料集佔總資料集的比例，預設為 0.2。

In [ ]:
X_a_path = 'Data/Xa'
X_b_path = 'Data/Xb'
Y_a_path = 'Data/Ya'
Y_b_path = 'Data/Yb'
X_a = bda.read_data(X_a_path)
X_a = bda.tune_data(X_a, 20000, 20000)
X_b = bda.read_data(X_b_path)
X_b = bda.tune_data(X_b, 20000, 20000)
Y_a = bda.read_data(Y_a_path)
Y_a = bda.tune_data(Y_a, 21000, 21000)
Y_b = bda.read_data(Y_b_path)
Y_b = bda.tune_data(Y_b, 21000, 21000)
path_Xa = 'outputs/BigData_CNN3D_Xa_coord.pth'
path_Xb = 'outputs/BigData_CNN3D_Xb_coord.pth'
path_Ya = 'outputs/BigData_CNN3D_Ya_coord.pth'
path_Yb = 'outputs/BigData_CNN3D_Yb_coord.pth'

In [60]:
bda.validation_skfold(X_a, 5, 'X', 'discrete', path_Xa)

      訓練     測試
第1次  1.0  1.000
第2次  1.0  0.980
第3次  1.0  0.980
第4次  1.0  0.980
第5次  1.0  1.000
平均   1.0  0.988


In [61]:
bda.validation_skfold(X_b, 5, 'X', 'discrete', path_Xb)

        訓練     測試
第1次  0.980  0.980
第2次  1.000  1.000
第3次  1.000  0.960
第4次  0.990  1.000
第5次  0.990  0.980
平均   0.992  0.984


In [62]:
bda.validation_skfold(Y_a, 5, 'Y', 'discrete',  path_Ya)

        訓練     測試
第1次  0.990  1.000
第2次  1.000  1.000
第3次  1.000  1.000
第4次  1.000  0.960
第5次  1.000  1.000
平均   0.998  0.992


In [64]:
bda.validation_skfold(Y_b, 5, 'Y', 'discrete',  path_Yb)

        訓練     測試
第1次  1.000  1.000
第2次  1.000  0.980
第3次  0.990  1.000
第4次  0.990  1.000
第5次  0.990  0.980
平均   0.994  0.992


In [3]:
del Y_a["data_220_26"]
del Y_a["data_220_52"]
del Y_b["data_220_26"]
del Y_b["data_220_52"]

In [5]:
bda.validation_skfold(X_a, 5, 'X', 'coord', path_Xa, inner = True)


各 label 的 MSE：
label 65 的 MSE = 3.87
label 80 的 MSE = 0.27
label 95 的 MSE = 0.18
label 130 的 MSE = 258.50

各 label 的 MSE：
label 65 的 MSE = 4.43
label 80 的 MSE = 7.53
label 95 的 MSE = 0.09
label 130 的 MSE = 2.75

各 label 的 MSE：
label 65 的 MSE = 3.21
label 80 的 MSE = 0.13
label 95 的 MSE = 0.12
label 130 的 MSE = 4.66

各 label 的 MSE：
label 65 的 MSE = 0.02
label 80 的 MSE = 0.20
label 95 的 MSE = 0.09
label 130 的 MSE = 9.45

各 label 的 MSE：
label 65 的 MSE = 1.44
label 80 的 MSE = 0.00
label 95 的 MSE = 28.91
label 130 的 MSE = 7.11
        訓練      測試
第1次  3.470  68.550
第2次  2.410   3.800
第3次  2.890   2.060
第4次  4.530   2.390
第5次  4.380   9.190
平均   3.536  17.198


In [6]:
bda.validation_skfold(X_b, 5, 'X', 'coord', path_Xb, inner = True)


各 label 的 MSE：
label 65 的 MSE = 10.26
label 80 的 MSE = 3.19
label 95 的 MSE = 0.21
label 130 的 MSE = 6.21

各 label 的 MSE：
label 65 的 MSE = 1.47
label 80 的 MSE = 0.43
label 95 的 MSE = 0.81
label 130 的 MSE = 24.49

各 label 的 MSE：
label 65 的 MSE = 3.14
label 80 的 MSE = 0.32
label 95 的 MSE = 0.85
label 130 的 MSE = 22.81

各 label 的 MSE：
label 65 的 MSE = 1.93
label 80 的 MSE = 0.47
label 95 的 MSE = 61.38
label 130 的 MSE = 5.47

各 label 的 MSE：
label 65 的 MSE = 0.13
label 80 的 MSE = 19.61
label 95 的 MSE = 0.44
label 130 的 MSE = 387.72
       訓練      測試
第1次  2.51   5.110
第2次  5.64   6.540
第3次  3.58   6.700
第4次  5.53  16.970
第5次  4.19  99.710
平均   4.29  27.006


In [7]:
bda.validation_skfold(Y_a, 5, 'Y', 'coord', path_Ya, inner = True)


各 label 的 MSE：
label 220 的 MSE = 1.34
label 260 的 MSE = 0.62
label 300 的 MSE = 0.16
label 380 的 MSE = 53.72

各 label 的 MSE：
label 220 的 MSE = 0.34
label 260 的 MSE = 0.11
label 300 的 MSE = 0.84
label 380 的 MSE = 76.32

各 label 的 MSE：
label 220 的 MSE = 0.80
label 260 的 MSE = 0.61
label 300 的 MSE = 0.46
label 380 的 MSE = 85.44

各 label 的 MSE：
label 220 的 MSE = 0.14
label 260 的 MSE = 0.32
label 300 的 MSE = 0.84
label 380 的 MSE = 66.35

各 label 的 MSE：
label 220 的 MSE = 3.07
label 260 的 MSE = 35.11
label 300 的 MSE = 0.06
label 380 的 MSE = 19.85
         訓練      測試
第1次   6.800  13.370
第2次  15.740  18.580
第3次   8.010  20.900
第4次   9.120  16.560
第5次   8.260  14.200
平均    9.586  16.722


In [8]:
bda.validation_skfold(Y_b, 5, 'Y', 'coord', path_Yb, inner = True)


各 label 的 MSE：
label 220 的 MSE = 14.50
label 260 的 MSE = 0.36
label 300 的 MSE = 7.53
label 380 的 MSE = 69.14

各 label 的 MSE：
label 220 的 MSE = 3.71
label 260 的 MSE = 5.11
label 300 的 MSE = 0.90
label 380 的 MSE = 30.78

各 label 的 MSE：
label 220 的 MSE = 1.53
label 260 的 MSE = 1.79
label 300 的 MSE = 20.21
label 380 的 MSE = 107.57

各 label 的 MSE：
label 220 的 MSE = 0.45
label 260 的 MSE = 1.43
label 300 的 MSE = 18.66
label 380 的 MSE = 187.23

各 label 的 MSE：
label 220 的 MSE = 1.40
label 260 的 MSE = 0.75
label 300 的 MSE = 5.78
label 380 的 MSE = 148.39
         訓練     測試
第1次  20.820  22.06
第2次  16.220   9.82
第3次  22.490  31.83
第4次  34.520  51.20
第5次  18.240  38.34
平均   22.458  30.65
